# BotDriver RTRS

Este notebook demonstra como utilizar a classe `BotDriver` para coletar e, opcionalmente, baixar os relatórios públicos de auditoria disponíveis no site da RTRS (Round Table on Responsible Soy).

## Pré-requisitos

- Google Chrome instalado e compatível com o `chromedriver`.
- Dependências Python listadas em `requirements.txt` (especialmente `selenium`, `webdriver-manager`, `beautifulsoup4`, `pandas`, `requests` e `tqdm`).
- Acesso à internet para baixar o WebDriver e os relatórios em PDF.

In [1]:
from pathlib import Path
import sys

current_dir = Path.cwd()
project_root = current_dir if (current_dir / 'scripts').exists() else current_dir.parent

if not (project_root / 'scripts').exists():
    raise RuntimeError("Não foi possível localizar o diretório 'scripts' a partir deste notebook.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f'Diretório raiz do projeto: {project_root}')

Diretório raiz do projeto: /home/cleyton/dev/msc_thesis_naea_ufpa


In [3]:
!pip isntall selenium

ERROR: unknown command "isntall" - maybe you meant "install"


In [4]:
from scripts.data_scrapping.botdriver import BotDriver
from IPython.display import display
from bs4 import BeautifulSoup
import pandas as pd
import time

# Ajuste as configurações abaixo conforme necessário
CHROME_OPTIONS = (
    '--headless=new',
    '--disable-gpu',
    '--no-sandbox',
    '--window-size=1920,1080',
)

BASE_URL = BotDriver.DEFAULT_BASE_URL
TARGET_COUNTRY = 'Brazil'
CSV_NAME = 'certified_producers_rtrs.csv'
DOWNLOAD_PDFS = False

ModuleNotFoundError: No module named 'selenium'

In [ ]:
links = []
bot = None
df = pd.DataFrame()

try:
    bot = BotDriver(*CHROME_OPTIONS, base_url=BASE_URL)
    bot.load_base_page()

    countries = bot.get_country_list()
    display(pd.DataFrame({'Países disponíveis': countries}))

    if TARGET_COUNTRY not in countries:
        raise ValueError(f"País '{TARGET_COUNTRY}' não está disponível no filtro.")

    bot.select_country(TARGET_COUNTRY)
    bot.click_search()

    time.sleep(2)
    html = BeautifulSoup(bot.browser.page_source, 'html.parser')
    audits_r_list = html.find_all('div', class_='zyxaudit-result-line')

    if not audits_r_list:
        print('Nenhum relatório encontrado. Ajuste os filtros e tente novamente.')
    else:
        df, links = bot.extract_general_info(audits_r_list, csv_name=CSV_NAME)
        display(df.head())
        print(f'Dados salvos em: {bot.external_data_folder / CSV_NAME}')

        if DOWNLOAD_PDFS:
            bot.download_all_reports(links)
finally:
    if bot is not None:
        bot.close()

links